[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronsong1234/GrandQC_IDC-Validation/blob/main/notebooks/04_reference_provenance.ipynb)


# 04 — Zenodo Reference-Provenance Reconstruction

**Purpose.** Test whether the authors' **public 2024 GrandQC code + released weights** reproduce the published Zenodo tissue boundaries when run on the original SVS. If not, the Zenodo masks are not a reproducible correctness reference.

**Inputs.** Original SVS (GDC); GrandQC `cpath-ukk/grandqc`@`84f3fec`; published checkpoints; Zenodo BRCA masks.

**Expected outputs.** `provenance_experiment/prov_5slide_results.json` + tissue-IoU table.

**Environment.** `environment.yml` / `requirements-lock.txt`. **Runtime.** ~10-20 min CPU (tissue stage, 5 slides).
**GPU.** not required.


### The three-way logic

- **authors' 2024 code vs Zenodo** — the reproducibility question.
- **our fork vs Zenodo** — does the IDC fork make it worse? (No.)
- **2024 code vs our fork** — are the two public runs the same? (Yes, ~0.99.)

If both public runs diverge from Zenodo by the same amount while agreeing with each other, the gap is release-wide, not a fork artifact. See `provenance_experiment/PROVENANCE_AUDIT.md` for the full manifest and setup (model SHA-256, strict state-dict load, device-only patch).

In [ ]:
# --- Colab bootstrap: install the grandqc_idc package and its deps ---
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("GrandQC_IDC-Validation"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/ronsong1234/GrandQC_IDC-Validation.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "./GrandQC_IDC-Validation[plot]", "pandas"], check=False)
else:
    sys.path.insert(0, os.path.abspath(".."))   # import grandqc_idc from the repo root

# NOTE: this notebook compares pre-generated masks. Produce them first with the authors'
# 2024 code + the fork (see provenance_experiment/PROVENANCE_AUDIT.md), then point the
# path variables below at them. It does not download slides itself.

In [ ]:
# Setup: clone the 2024 authors' code at commit 84f3fec and run tissue detection
# on the original SVS. See PROVENANCE_AUDIT.md for the exact commands; this cell
# assumes masks have been generated into the paths below.
import sys, os, glob, json
import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

AUTHORS_2024_MASKS = 'provenance_masks/2024'   # tissue masks from 84f3fec
FORK_MASKS        = 'provenance_masks/fork'     # fork tissue masks
ZENODO_MASKS      = 'provenance_masks/zenodo'   # Zenodo reference masks


In [ ]:
def up(m, shape):
    return np.array(Image.fromarray(m).resize(shape[::-1], Image.Resampling.NEAREST))
def iou(a, b):
    u = (a | b).sum(); return float((a & b).sum() / u) if u else 1.0

rows = []
for zpath in sorted(glob.glob(f'{ZENODO_MASKS}/*_mask.png')):
    bc = os.path.basename(zpath).split('.')[0]
    zen = np.array(Image.open(zpath)); zt = (zen != 7) & (zen != 0)
    fork = np.array(Image.open(f'{FORK_MASKS}/{bc}_MASK.png'))
    m24 = np.array(Image.open(glob.glob(f'{AUTHORS_2024_MASKS}/{bc}*_MASK.png')[0]))
    th, tw = fork.shape
    m24c = m24[-th:, -tw:] if m24.shape != fork.shape else m24  # small-thumb crop
    t24, tf = up(m24c, zen.shape) == 0, up(fork, zen.shape) == 0
    rows.append({'slide': bc, 'authors2024_vs_zenodo': round(iou(t24, zt), 4),
                 'fork_vs_zenodo': round(iou(tf, zt), 4),
                 'authors2024_vs_fork': round(iou(t24, tf), 4)})
import pandas as pd
df = pd.DataFrame(rows); print(df.to_string(index=False))
json.dump(rows, open('../provenance_experiment/prov_5slide_results.json', 'w'), indent=1)

**Result:** authors-2024 and fork agree at 0.986–0.998 on every slide, while both sit at ~0.70 vs Zenodo on rim slides. The public release does not reproduce the Zenodo masks. Next step: `provenance_experiment/AUTHORS_NOTE.md`.